# 1. Imports and Initiate

## 1.1 Libraries

In [ ]:
import os
os.chdir("C:\\Users\\phys-pico-lab\\Documents\\Jupyter_Measurements")
print(os.getcwd())

In [ ]:
%config IPCompleter.greedy=True

# convenience import for all QCCS software functionality
from laboneq.simple import *

# helper import - needed to extract qubit and readout parameters from measurement data
from lib.helpers.example_notebook_helper import *
from lib.helpers.meas_helper_mod import *
from lib.helpers.setup_helper import *
from lib.helpers.save_data_helper import *
from lib.helpers.create_meas_helper import *
#from helpers.pop_temp_helper import *
from lib.helpers.pop_temp_helper_v2 import *
from lib.helpers.fitting_helper import *
import lib.utils.calculator as calc

#import from python librarys
from scipy.io import savemat, loadmat
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy import fft
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os.path
import time
import json as js
from scresonators.wrapper_functions import fit_single_STS_wrapper, plot_single_STS_wrapper
import pyvisa

#import instruments drivers
from blueftc.BlueforsController import BlueFTController
from lib.devices.bftc_credentials import BFTS_API_KEY, BFTS_IP, PORT_NUMBER, MXC_ID
from lib.devices.KeysightDMM34465A import KeysightDMM34465A
from lib.devices.SIM_wrapper import SIM900, SIM928
from lib.devices.YokoGS200_wrapper import YokoGS200

#imports for XLD server measurements
from lib.devices.XLD_Server_Client import XLDMeasClient
from time import sleep
from lib.devices.XLD_Server_Passkey import xld_ip

from qmeas.qsample import QSample
from qmeas.qparameters import QBaseParameters, QLinkedParameters

In [ ]:
from playsound import playsound

def meas_ready():
    if not mute:
        playsound(os.path.join('lib', 'utils', 'ding.mp3'))

mute = False

## 1.2 Setup temperature controller

In [ ]:
#Temperature controller parameters

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

BFTCont = BlueFTController(ip=BFTS_IP, port=PORT_NUMBER, key=BFTS_API_KEY, 
                              mixing_chamber_channel_id=MXC_ID)

BFTCont.get_mxc_temperature()

## 1.3 Voltage source setup
*Here flux fixed at $-1.2\text{V}$ via $5.6k\Omega$*

In [ ]:
#Volt source parameters
Yoko_addr = 'TCPIP0::192.168.1.201::inst0::INSTR'
volt_source = YokoGS200(address=Yoko_addr)
R = 5.4e3 #Ohm

In [ ]:
volt_source.set_range(10)
volt_source.get_range()

In [ ]:
# temp_dict = BFTCont.get_channel_temps_in_time(5, 57600)

# # print(temp_dict.keys())

# temps_ch_5 = pd.DataFrame({'time': temp_dict['timestamp'], 
#                            'temp': temp_dict['temperature']})
# temps_ch_5.to_csv('channel_5_temp_data_2024-07-11_23-00--2024-07-12_14-38.txt', index=False)

## 1.4 Sample parameters

In [ ]:
cooldown_nr = 2

#dictionary for sample parameters
sample_parameters = {
    'folder_name': r'N:\xld\Kubatkin\Data',
    'sample_name': r'S3',
    'structure_name': 'Q2'
}

qsample_params = QSample(directory=sample_parameters['folder_name'], 
                         sample=sample_parameters['sample_name'], 
                         structure=sample_parameters['structure_name'])

## 1.5 Qubit and LO parameters

### A - New parameters

In [ ]:
# a collection of qubit control and readout parameters as a python dictionary
qubit_parameters = {
    'ro_freq' : 136e6, # 9.320e9,     # readout frequency of qubit 0 in [Hz] - relative to local oscillator for readout drive upconversion
    'ro_amp' : 0.5,              # readout amplitude
    'ro_amp_spec': 0.05,          # readout amplitude for spectroscopy 0.05
    'ro_len' : 2.0e-6,           # readout pulse length in [s]
    'ro_len_spec' : 4.0e-6,      # readout pulse length for resonator spectroscopy in [s]
    'ro_delay': 0.0,           # readout delay after last drive signal in [s]
    'ro_int_delay' : 330e-9,     # readout line offset calibration - delay between readout pulse and start of signal acquisition in [s]
    
    'th_res_freq': 0.0,
    
    'qb_freq': 50e6,            # qubit 0 drive frequency in [Hz] - relative to local oscillator for qubit drive upconversion
    'qb_anharm': 200e6,
    'qb_amp_spec': 0.0001,       # drive amplitude of qubit spectroscopy
    'qb_len_spec': 15e-6,        # drive pulse length for qubit spectroscopy in [s]
    'qb_len' : 6e-8,             # qubit drive pulse length in [s]
    'pi_amp' : 1.0,              # qubit drive amplitude for pi pulse
    'pi_half_amp' : 0.25,        # qubit drive amplitude for pi/2 pulse
    'qb_t1' : 6e-6,              # qubit T1 time
    'qb_t2_ramsey' : 1e-6,       # qubit T2 time
    'ramsey_det' : 2e6,       # qubit frequency detuning relative to qb_freq in [Hz]
    'qb_t2_echo'   : 1e-6,       # qubit T2 time
    'relax' : 500e-6             # delay time after each measurement for qubit reset in [s]
}

qubit_parameters = QBaseParameters(sample=qsample_params,
                                    name=f'qubit_parameters_cooldown_{cooldown_nr}',
                                    parameters=qubit_parameters)

In [ ]:
# up / downconversion settings - to convert between IF and RF frequencies 
# steps of 200Mhz
lo_settings = {
    'qb_lo': 3.8e9,              # qubit LO frequency in [Hz]
    'ro_lo': 6.2e9              # R1 readout LO frequency in [Hz]
}

lo_settings = QBaseParameters(sample=qsample_params,
                             name=f'lo_settings_cooldown_{cooldown_nr}',
                             parameters=lo_settings)

### B - Load parameters from file

In [ ]:
# #path to file with qubit parameters
# qp_file_path = r'N:\xld\Qubit Termometry\Data\S411-21\Q1\2025-04-07 Jupiter\Qubit_params_BACK15_41_28.txt'

# #path to file with lo settings
# lo_file_path = r'N:\xld\Qubit Termometry\Data\S351-23\QT\2023-12-14 Jupiter\LO_settings_dict15_19_08.txt'

# #qubit parameters loading
# with open(qp_file_path, 'r') as fp:
#     qubit_parameters = js.load(fp)
    
# #lo settings loading
# with open(lo_file_path, 'r') as fp:
#     lo_settings = js.load(fp)

previous_cooldown = 1

root_dir = os.path.join(sample_parameters['folder_name'], sample_parameters['sample_name'], sample_parameters['structure_name'], 'parameters')

with open(os.path.join(root_dir, f'lo_settings_cooldown_{previous_cooldown}.txt'), 'r') as f:
    pars = js.load(f)

    lo_settings = QBaseParameters(sample=qsample_params,
                            name=f'lo_settings_cooldown_{cooldown_nr}',
                            parameters=pars)

with open(os.path.join(root_dir, f'qubit_parameters_cooldown_{previous_cooldown}.txt'), 'r') as f:
    pars = js.load(f)

    qubit_parameters = QBaseParameters(sample=qsample_params,
                            name=f'qubit_parameters_cooldown_{cooldown_nr}',
                            parameters=pars)

# with open(os.path.join(root_dir, f'twpa_settings_cooldown_{previous_cooldown}.txt'), 'r') as f:
#     pars = js.load(f)

#     twpa_settings = QBaseParameters(sample=qsample_params,
#                             name=f'twpa_settings_cooldown_{this_cooldown}',
#                             parameters=pars)

## 1.6 Zurich Instruments SHFQC Setup

In [ ]:
# define the DeviceSetup from descriptor - additionally include information on the dataserver used to connect to the instruments 
my_setup = DeviceSetup.from_descriptor(
    my_descriptor,
    server_host="localhost",
    server_port="8004",
    setup_name="XLD",
) 

# define Calibration object based on qubit control and readout parameters
my_calibration = define_calibration(parameters=qubit_parameters, lo_settings=lo_settings)

# apply calibration to device setup
my_setup.set_calibration(my_calibration)

In [ ]:
## define shortcut to logical signals for convenience
lsg_q0 = my_setup.logical_signal_groups["q0"].logical_signals
drive_Oscillator_q0 = lsg_q0["drive_line"].oscillator
drive_ef_Oscillator_q0 = lsg_q0["drive_ef_line"].oscillator
readout_Oscillator_q0 = lsg_q0["measure_line"].oscillator
acquire_Oscillator_q0 = lsg_q0["acquire_line"].oscillator

lsg_q0["acquire_line"].port_delay

## 1.7 Create and Connect to a LabOne Q Session

In [ ]:
# perform experiments in emulation mode only? - if True, also generate dummy data for fitting
emulate = False

# create and connect to a session
my_session = Session(device_setup=my_setup)
my_session.connect(do_emulation=emulate, ignore_version_mismatch = True)

lsg_q0["acquire_line"].range = -35

In [ ]:
#Check ranges
print('Drive line range:', lsg_q0["drive_line"].range, 'dBm')
print('Measure line range:', lsg_q0["measure_line"].range, 'dBm')
print('Acquire line range:', lsg_q0["acquire_line"].range, 'dBm')